<a href="https://colab.research.google.com/github/Varun18-hub-dev/Varun18-hub-dev-Indian-legal-judgement-classification/blob/main/NLP_trained_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
import numpy as np
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier



In [41]:
data = pd.read_csv("/content/drive/MyDrive/Indian_law_NLP/case_files_total.csv", engine='python')

In [42]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53446 entries, 0 to 53445
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      53446 non-null  int64  
 1   name            53445 non-null  object 
 2   case_category   43612 non-null  object 
 3   case_type       52710 non-null  object 
 4   case_info       53421 non-null  object 
 5   judgement       50144 non-null  object 
 6   tokens          53444 non-null  float64
 7   sentences       53444 non-null  float64
 8   label           53444 non-null  object 
 9   proof_sentence  53444 non-null  object 
dtypes: float64(2), int64(1), object(7)
memory usage: 4.1+ MB


In [43]:
data.head(5)

,Unnamed: 0,name,case_category,case_type,case_info,judgement,tokens,sentences,label,proof_sentence
0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,74634371.txt,civil,writ petition,reportable in the supreme court of india civil...,thus spake benjamin franklin in his letter of ...,9974.0,254.0,Accepted,"34. we are, therefore, of the view that this a..."
2,2,746230.txt,civil,writ petition,case number appeal civil 588 of 1972 petitio...,difficult to share the view taken by the high...,1411.0,56.0,Accepted,"6. accordingly, we allow the appeal with costs."
3,3,746330.txt,civil,writ petition,petitioner western india plywood limited vs. r...,j u d g m e n t the following judgement of the...,2616.0,85.0,Accepted,for the aforesaid reasons this appeal is allowed.
4,4,746327.txt,civil,writ petition,"petitioner asstt.\ncustodian, e.p. ors.\nvs. r...",custodian first stated that the property was n...,2763.0,95.0,Accepted,appeal allowed.


In [54]:
(data.isnull().sum()/len(data))*100

,0
case_category,0.0
judgement,0.0


In [46]:
data['case_category'].fillna(data['case_category'].mode(),inplace=True)


In [47]:
data = data.drop(columns=['name','case_type','tokens','sentences','label','proof_sentence'])

In [48]:
data = data.drop(columns=['Unnamed: 0'])

In [49]:
data = data.drop(columns=['case_info'])

In [50]:
data.columns

Index(['case_category', 'judgement'], dtype='object')

In [51]:
data.head()

,case_category,judgement
0,civil,NaN
1,civil,thus spake benjamin franklin in his letter of ...
2,civil,difficult to share the view taken by the high...
3,civil,j u d g m e n t the following judgement of the...
4,civil,custodian first stated that the property was n...


In [53]:
data['judgement'].fillna(data['judgement'].mode()[0],inplace=True)

In [55]:
#clean text
def clean_text(text):
  text = text.lower()
  text = re.sub(r'[^a-zA-Z]',' ',text)
  text = re.sub(r'\s+',' ',text)
  text = text.strip()

  nltk.download('stopwords')
  nltk.download('wordnet')

  stop_words = set(stopwords.words('english'))
  lemmatizer = WordNetLemmatizer()

  words = text.split()

  new_words = []

  for w in words:
    if w not in stop_words:
      new_words.append(w)

  words = new_words

  new_words = []

  for w in words:
    new_words.append(lemmatizer.lemmatize(w))
  words = new_words
  return " ".join(words)




In [56]:
data['clean_category'] = data['case_category'].apply(clean_text)

Streaming output truncated to the last 5000 lines.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] D

In [57]:
data['clean_judgement'] = data['judgement'].apply(clean_text)

Streaming output truncated to the last 5000 lines.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] D

In [58]:
data.head()

,case_category,judgement,clean_category,clean_judgement
0,civil,"respondents judgment dr. arijit pasayat, j. 1....",civil,respondent judgment dr arijit pasayat j challe...
1,civil,thus spake benjamin franklin in his letter of ...,civil,thus spake benjamin franklin letter november j...
2,civil,difficult to share the view taken by the high...,civil,difficult share view taken high court paragrap...
3,civil,j u d g m e n t the following judgement of the...,civil,j u g e n following judgement court delivered ...
4,civil,custodian first stated that the property was n...,civil,custodian first stated property evacuee proper...


In [59]:
data = data.drop(columns=['case_category','judgement'])

In [60]:
X = data['clean_judgement']
y = data['clean_category']

In [61]:
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(X)

le = LabelEncoder()
y = le.fit_transform(y)

In [62]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [63]:
model = LogisticRegression()

In [64]:
model.fit(X_train,y_train)
y_pred = model.predict(X_test)

In [65]:
accuracy = accuracy_score(y_test,y_pred)
print("Accuracy: ",accuracy)
print("classification report : ",classification_report(y_test,y_pred))
print("confusion matrix : ", confusion_matrix(y_test,y_pred))

Accuracy:  0.9069223573433115
classification report :                precision    recall  f1-score   support

           0       0.94      0.95      0.94      8877
           1       0.74      0.70      0.72      1813

    accuracy                           0.91     10690
   macro avg       0.84      0.82      0.83     10690
weighted avg       0.90      0.91      0.91     10690

confusion matrix :  [[8432  445]
 [ 550 1263]]


In [66]:
model_2 = RandomForestClassifier()
model_2.fit(X_train,y_train)
y_pred_2 = model_2.predict(X_test)

accuracy_2 = accuracy_score(y_test,y_pred_2)
print("Accuracy: ",accuracy_2)

Accuracy:  0.94911131898971


In [67]:
model_3 = DecisionTreeClassifier()
model_3.fit(X_train,y_train)
y_pred_3 = model_3.predict(X_test)

accuracy_3 = accuracy_score(y_test,y_pred_3)
print("Accuracy: ",accuracy_3)

Accuracy:  0.9569691300280636
